# 00 - Preparação do SQL Server

Este notebook cria o banco LojaDB e carrega dados fictícios de uma loja.

| Tabela | Conteúdo |
|---|---|
| categorias | Categorias dos produtos |
| produtos | Produtos, preços e estoque |
| clientes | Clientes fictícios |
| pedidos | Pedidos realizados pelos clientes |
| itens_pedido | Produtos e quantidades de cada pedido |

Os dados foram criados para este trabalho. Uma categoria possui vários
produtos; um cliente pode realizar vários pedidos; cada pedido possui
itens associados aos produtos.

O SQL Server utiliza chaves primárias e estrangeiras para representar
essas relações.

## 1. Configuração e conexão

Carregamos as configurações do arquivo .env e conectamos ao SQL Server.
As senhas ficam no arquivo local, ignorado pelo Git.

In [1]:
import os
from pathlib import Path
import pymssql
from dotenv import load_dotenv

raiz = Path.cwd()
if not (raiz / "pyproject.toml").is_file():
    raiz = raiz.parent

load_dotenv(raiz / ".env", override=True)

conexao_master = pymssql.connect(
    server=os.environ["SQLSERVER_HOST"],
    port=os.environ["SQLSERVER_PORT"],
    user=os.environ["SQLSERVER_USER"],
    password=os.environ["SQLSERVER_PASSWORD"],
    database="master",
    charset="UTF-8",
    autocommit=True,
    login_timeout=15,
    timeout=60
)

cursor_master = conexao_master.cursor()
cursor_master.execute(
    "IF DB_ID(N'LojaDB') IS NULL CREATE DATABASE LojaDB"
)
cursor_master.close()
conexao_master.close()

conexao = pymssql.connect(
    server=os.environ["SQLSERVER_HOST"],
    port=os.environ["SQLSERVER_PORT"],
    user=os.environ["SQLSERVER_USER"],
    password=os.environ["SQLSERVER_PASSWORD"],
    database="LojaDB",
    charset="UTF-8",
    autocommit=True,
    login_timeout=15,
    timeout=60
)
cursor = conexao.cursor()
print("Conectado ao banco LojaDB.")

Conectado ao banco LojaDB.


## 2. Criação das tabelas — DDL

Para repetir a carga, removemos somente as cinco tabelas deste exemplo,
começando pelas que dependem das outras. Depois recriamos sua estrutura.

DECIMAL(10,2) representa valores monetários com duas casas decimais.
As chaves estrangeiras vinculam produtos, clientes, pedidos e itens.

In [2]:
cursor.execute("""
    DROP TABLE IF EXISTS dbo.itens_pedido;
    DROP TABLE IF EXISTS dbo.pedidos;
    DROP TABLE IF EXISTS dbo.produtos;
    DROP TABLE IF EXISTS dbo.clientes;
    DROP TABLE IF EXISTS dbo.categorias;

    CREATE TABLE dbo.categorias (
        id_categoria INT PRIMARY KEY,
        nome_categoria NVARCHAR(100) NOT NULL
    );

    CREATE TABLE dbo.produtos (
        id_produto INT PRIMARY KEY,
        nome_produto NVARCHAR(100) NOT NULL,
        preco DECIMAL(10,2) NOT NULL,
        estoque INT NOT NULL,
        id_categoria INT NOT NULL,
        FOREIGN KEY (id_categoria)
            REFERENCES dbo.categorias(id_categoria)
    );

    CREATE TABLE dbo.clientes (
        id_cliente INT PRIMARY KEY,
        nome_cliente NVARCHAR(100) NOT NULL,
        email NVARCHAR(150) NOT NULL
    );

    CREATE TABLE dbo.pedidos (
        id_pedido INT PRIMARY KEY,
        id_cliente INT NOT NULL,
        data_pedido DATE NOT NULL,
        status_pedido NVARCHAR(30) NOT NULL,
        FOREIGN KEY (id_cliente)
            REFERENCES dbo.clientes(id_cliente)
    );

    CREATE TABLE dbo.itens_pedido (
        id_item INT PRIMARY KEY,
        id_pedido INT NOT NULL,
        id_produto INT NOT NULL,
        quantidade INT NOT NULL,
        preco_unitario DECIMAL(10,2) NOT NULL,
        FOREIGN KEY (id_pedido)
            REFERENCES dbo.pedidos(id_pedido),
        FOREIGN KEY (id_produto)
            REFERENCES dbo.produtos(id_produto)
    );
""")
print("Cinco tabelas criadas.")

Cinco tabelas criadas.


## 3. Carga dos dados

Inserimos duas categorias, dez produtos, cinco clientes, cinco pedidos
e dez itens de pedido. Os produtos mantêm o cenário do trabalho anterior.

O preço unitário do item registra o preço utilizado naquele pedido.

In [3]:
cursor.execute("""
    INSERT INTO dbo.categorias VALUES
        (1, N'Informática'),
        (2, N'Papelaria');

    INSERT INTO dbo.produtos VALUES
        (1, N'Mouse', 49.90, 20, 1),
        (2, N'Teclado', 120.00, 10, 1),
        (3, N'Caderno', 19.90, 30, 2),
        (4, N'Caneta', 3.50, 100, 2),
        (5, N'Headset', 199.90, 12, 1),
        (6, N'Webcam', 149.90, 8, 1),
        (7, N'Pendrive', 39.90, 25, 1),
        (8, N'Lapiseira', 9.90, 40, 2),
        (9, N'Borracha', 2.50, 60, 2),
        (10, N'Marcador', 6.90, 35, 2);

    INSERT INTO dbo.clientes VALUES
        (1, N'Ana Souza', 'ana@example.com'),
        (2, N'Bruno Lima', 'bruno@example.com'),
        (3, N'Carla Santos', 'carla@example.com'),
        (4, N'Diego Alves', 'diego@example.com'),
        (5, N'Elisa Costa', 'elisa@example.com');

    INSERT INTO dbo.pedidos VALUES
        (1, 1, '2026-10-01', N'Concluído'),
        (2, 2, '2026-10-02', N'Concluído'),
        (3, 3, '2026-10-03', N'Em preparação'),
        (4, 4, '2026-10-04', N'Concluído'),
        (5, 5, '2026-10-05', N'Em preparação');

    INSERT INTO dbo.itens_pedido VALUES
        (1, 1, 1, 1, 49.90),
        (2, 1, 2, 1, 120.00),
        (3, 2, 3, 2, 19.90),
        (4, 2, 8, 1, 9.90),
        (5, 3, 5, 1, 199.90),
        (6, 3, 6, 1, 149.90),
        (7, 4, 7, 2, 39.90),
        (8, 4, 9, 3, 2.50),
        (9, 5, 10, 2, 6.90),
        (10, 5, 3, 1, 19.90);
""")
print("Dados fictícios inseridos.")

Dados fictícios inseridos.


## 4. Validação

Consultamos as tabelas do banco e verificamos suas quantidades.
Esta será a origem da extração para o MinIO.

In [4]:
cursor.execute("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'dbo' AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME
""")
tabelas = [linha[0] for linha in cursor.fetchall()]

esperado = {
    "categorias": 2,
    "produtos": 10,
    "clientes": 5,
    "pedidos": 5,
    "itens_pedido": 10
}

assert set(tabelas) == set(esperado), "Confira as tabelas do banco."

for tabela in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM dbo.[{tabela}]")
    quantidade = cursor.fetchone()[0]
    print(f"{tabela}: {quantidade} registros")
    assert quantidade == esperado[tabela]

cursor.close()
conexao.close()
print("Validação concluída: cinco tabelas preenchidas.")

categorias: 2 registros
clientes: 5 registros
itens_pedido: 10 registros
pedidos: 5 registros
produtos: 10 registros
Validação concluída: cinco tabelas preenchidas.
